In [ ]:
import os
os.chdir("D:/Projects/volatility-radar")

In [ ]:
import numpy as np
import pandas as pd

eurusd = pd.read_csv("data/processed/eurusd_forex.csv")
eurusd.head()

In [ ]:
eurusd.columns

In [ ]:
eurusd.dtypes

In [ ]:
eurusd['date'] = pd.to_datetime(eurusd['date'])
eurusd.dtypes

In [ ]:
def convert(x):
    if x == "" or x.lower() == 'nan':
        return np.nan
    x = x.replace("<", "").replace(">", "").replace("~", "")
    x = x.upper()
    if "<" in x:
        return float(x.replace("<", "")) * 0.99 
    if ">" in x:
        return float(x.replace(">", "")) * 1.01
    if "%" in x:
        return(float(x.replace("%", "")) / 100)
    if "K" in x:
        return(float(x.replace("K", "")) * 1e+3)
    if "M" in x:
        return(float(x.replace("M", "")) * 1e+6)
    if "B" in x:
        return(float(x.replace("B", "")) * 1e+9)
    if "T" in x:
        return(float(x.replace("T", ""))* 1e+12)
    return float(x)

def parse_value(x):
    if pd.isna(x):
        return [np.nan, np.nan]

    x = x.strip()

    if "|" in x:
        parts = [p.strip() for p in x.split('|')]
        return [convert(parts[0]), convert(parts[1])]

    return [convert(x), np.nan]

eurusd[['actual_1', 'actual_2']] = eurusd['actual'].apply(parse_value).apply(pd.Series)
eurusd[['previous_1', 'previous_2']] = eurusd['previous'].apply(parse_value).apply(pd.Series)

eurusd.head(20)


In [ ]:
import os
os.chdir("D:/Projects/volatility-radar")

import re
import numpy as np
import pandas as pd

def build_calendar_features(pair):
    df = pd.read_csv(f"data/processed/{pair}_forex.csv")

    df['date'] = pd.to_datetime(df['date'])

    def convert(x):
        if pd.isna(x):
            return np.nan

        x = str(x).strip().upper()

        if x == "" or x.lower() == 'nan':
            return np.nan

        multiplier = 1.0
        if "<" in x:
            multiplier = 0.99
        elif ">" in x:
            multiplier = 1.01

        x = re.sub(r"[<>~]", "", x)

        try:
            if "%" in x:
                return float(x.replace("%", "")) / 100 * multiplier

            if "K" in x:
                return float(x.replace("K", "")) * 1e3 * multiplier

            if "M" in x:
                return float(x.replace("M", "")) * 1e6 * multiplier

            if "B" in x:
                return float(x.replace("B", "")) * 1e9 * multiplier

            if "T" in x:
                return float(x.replace("T", "")) * 1e12 * multiplier

            return float(x) * multiplier

        except:
            return np.nan

    def parse_value(x):
        if pd.isna(x):
            return [np.nan, np.nan]

        x = str(x).strip()

        if "|" in x:
            parts = [p.strip() for p in x.split('|')]
            return [convert(parts[0]), convert(parts[1])]

        return [convert(x), np.nan]

    df[['actual_1', 'actual_2']] = df['actual'].apply(parse_value).apply(pd.Series)
    df[['previous_1', 'previous_2']] = df['previous'].apply(parse_value).apply(pd.Series)

    df['has_second'] = df['actual_2'].notna().astype(int)

    df['change_1'] = df['actual_1'] - df['previous_1']
    df['change_2'] = df['actual_2'] - df['previous_2']

    df['change_rel_1'] = df['change_1'] / df['previous_1'].abs()
    df['change_rel_2'] = df['change_2'] / df['previous_2'].abs()

    df['change_rel_1'] = df['change_rel_1'].replace([np.inf, -np.inf], np.nan)
    df['change_rel_2'] = df['change_rel_2'].replace([np.inf, -np.inf], np.nan)

    df['z_score_1'] = df.groupby('event')['change_rel_1'].transform(lambda x: (x - x.mean()) / x.std())
    df['z_score_2'] = df.groupby('event')['change_rel_2'].transform(lambda x: (x - x.mean()) / x.std())

    impact_map = {'low':1, 'medium':2, 'high':3}
    df['impact_num'] = df['impact'].map(impact_map)

    df['signal_1'] = df['z_score_1'] * df['impact_num']
    df['signal_2'] = df['z_score_2'] * df['impact_num']

    df['signal_2'] = df['signal_2'].fillna(0)

    df['final_signal'] = df['signal_1'] + df['signal_2']

    df['direction'] = np.sign(df['final_signal'])

    return df

eurusd = build_calendar_features('eurusd')
eurusd.head()






In [48]:
import os
os.chdir("D:/Projects/volatility-radar")

import re
import numpy as np
import pandas as pd

def build_calendar_features(pair):
    df = pd.read_csv(f"data/processed/{pair}_forex.csv")

    df['date'] = pd.to_datetime(df['date'])

    def convert(x):
        if pd.isna(x):
            return np.nan

        multiplier = 1.0
        if "<" in x:
            multiplier = 0.99
        elif ">" in x:
            multiplier = 1.01

        x = str(x).strip().upper()

        x = re.sub(r"[<>~]", "", x)

        if x == "" or x.lower() == 'nan':
            return np.nan

        if re.match(r'^\d+-\d+-\d+$', x):
            votes = [float(v) for v in x.split("-")]  # Convert to float immediately
            x = (votes[0] - votes[1]) / (votes[0] + votes[1] + votes[2])
            return x

        try:
            if "%" in x:
                return float(x.replace("%", "")) / 100 * multiplier

            if "K" in x:
                return float(x.replace("K", "")) * 1e3 * multiplier

            if "M" in x:
                return float(x.replace("M", "")) * 1e6 * multiplier

            if "B" in x:
                return float(x.replace("B", "")) * 1e9 * multiplier

            if "T" in x:
                return float(x.replace("T", "")) * 1e12 * multiplier

            return float(x) * multiplier

        except:
            return np.nan

    def parse_value(x):
        if pd.isna(x):
            return [np.nan, np.nan]

        x = str(x).strip()

        if "|" in x:
            parts = [p.strip() for p in x.split('|')]
            return convert(parts[0])

        return convert(x)

    df['actual_clean'] = df['actual'].apply(parse_value)
    df['previous_clean'] = df['previous'].apply(parse_value)

    df['change'] = df['actual_clean'] - df['previous_clean']

    df['change_rel'] = df['change'] / df['previous_clean'].abs()

    df['change_rel'] = df['change_rel'].replace([np.inf, -np.inf], 0)
    df['change_rel'] = df['change_rel'].fillna(0)

    df['z_score'] = df.groupby('event')['change_rel'].transform(lambda x: (x - x.mean()) / x.std())

    impact_map = {'Low':1, 'Medium':2, 'High':3}
    df['impact_num'] = df['impact'].map(impact_map)

    df['signal'] = df['z_score'] * df['impact_num']

    return df

eurusd = build_calendar_features('eurusd')
eurusd.head(5)


,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal
0,2021-01-04,1:45pm,EUR,Spanish Manufacturing PMI,Low,51.0,49.8,51.0,49.8,1.2,0.024096,0.688935,1,0.688935
1,2021-01-04,2:15pm,EUR,Italian Manufacturing PMI,Low,52.8,51.5,52.8,51.5,1.3,0.025243,0.729711,1,0.729711
2,2021-01-04,2:20pm,EUR,French Final Manufacturing PMI,Low,51.1,51.1,51.1,51.1,0.0,0.000000,-0.352682,1,-0.352682
3,2021-01-04,2:25pm,EUR,German Final Manufacturing PMI,Low,58.3,58.6,58.3,58.6,-0.3,-0.005119,-0.748475,1,-0.748475
4,2021-01-04,2:30pm,EUR,Final Manufacturing PMI,Low,55.2,55.5,55.2,55.5,-0.3,-0.005405,-1.070335,1,-1.070335


In [49]:
eurusd.dtypes

date              datetime64[us]
time                         str
currency                     str
event                        str
impact                       str
actual                       str
previous                     str
actual_clean             float64
previous_clean           float64
change                   float64
change_rel               float64
z_score                  float64
impact_num                 int64
signal                   float64
dtype: object

In [50]:
eurusd.isnull().sum()

date              0
time              0
currency          0
event             0
impact            0
actual            0
previous          0
actual_clean      0
previous_clean    0
change            0
change_rel        0
z_score           0
impact_num        0
signal            0
dtype: int64

In [51]:
df = eurusd[eurusd['change_rel'].isna()]
df.head(15)

,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal


In [58]:
gbpusd = build_calendar_features('gbpusd')

In [59]:
gbpusd = build_calendar_features('gbpusd')
gbpusd.isnull().sum()

date              0
time              0
currency          0
event             0
impact            0
actual            0
previous          0
actual_clean      0
previous_clean    0
change            0
change_rel        0
z_score           8
impact_num        0
signal            8
dtype: int64

In [60]:
gbpusd[gbpusd['z_score'].isnull()].reset_index()

,index,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal
0,134,2021-02-04,5:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN
1,312,2021-03-18,3:30pm,GBP,Asset Purchase Facility,Low,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,1,NaN
2,500,2021-05-06,4:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN
3,699,2021-06-24,4:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN
4,864,2021-08-05,4:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN
5,1059,2021-09-23,4:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN
6,1230,2021-11-04,5:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN
7,1410,2021-12-16,5:30pm,GBP,Asset Purchase Facility,High,875B,875B,8.750000e+11,8.750000e+11,0.0,0.0,NaN,3,NaN


In [62]:
df = pd.read_csv("data/processed_v2/calendar_features.csv")
df.isnull().sum()

date              0
time              0
currency          0
event             0
impact            0
actual            0
previous          0
actual_clean      0
previous_clean    0
change            0
change_rel        0
z_score           0
impact_num        0
signal            0
dtype: int64

In [63]:
usdjpy = build_calendar_features('usdjpy')
usdjpy.isnull().sum()

date              0
time              0
currency          0
event             0
impact            0
actual            0
previous          0
actual_clean      0
previous_clean    0
change            0
change_rel        0
z_score           0
impact_num        0
signal            0
dtype: int64

In [1]:
import os
os.chdir("D:/Projects/volatility-radar")

import re
import numpy as np
import pandas as pd

def build_calendar_features(pair):
    df = pd.read_csv(f"data/processed/{pair}_forex.csv")

    df['date'] = pd.to_datetime(df['date'])

    def convert(x):
        if pd.isna(x):
            return np.nan

        multiplier = 1.0
        if "<" in x:
            multiplier = 0.99
        elif ">" in x:
            multiplier = 1.01

        x = str(x).strip().upper()

        x = re.sub(r"[<>~]", "", x)

        if x == "" or x.lower() == 'nan':
            return np.nan

        if re.match(r'^\d+-\d+-\d+$', x):
            votes = [float(v) for v in x.split("-")]  # Convert to float immediately
            x = (votes[0] - votes[1]) / (votes[0] + votes[1] + votes[2])
            return x

        try:
            if "%" in x:
                return float(x.replace("%", "")) / 100 * multiplier

            if "K" in x:
                return float(x.replace("K", "")) * 1e3 * multiplier

            if "M" in x:
                return float(x.replace("M", "")) * 1e6 * multiplier

            if "B" in x:
                return float(x.replace("B", "")) * 1e9 * multiplier

            if "T" in x:
                return float(x.replace("T", "")) * 1e12 * multiplier

            return float(x) * multiplier

        except:
            return np.nan

    def parse_value(x):
        if pd.isna(x):
            return [np.nan, np.nan]

        x = str(x).strip()

        if "|" in x:
            parts = [p.strip() for p in x.split('|')]
            return convert(parts[0])

        return convert(x)

    df['actual_clean'] = df['actual'].apply(parse_value)
    df['previous_clean'] = df['previous'].apply(parse_value)

    df['change'] = df['actual_clean'] - df['previous_clean']

    df['change_rel'] = df['change'] / df['previous_clean'].abs()

    df['change_rel'] = df['change_rel'].replace([np.inf, -np.inf], 0)
    df['change_rel'] = df['change_rel'].fillna(0)

    df['z_score'] = df.groupby('event')['change_rel'].transform(lambda x: (x - x.mean()) / x.std())
    df['z_score'] = df['z_score'].fillna(0)

    impact_map = {'Low':1, 'Medium':2, 'High':3}
    df['impact_num'] = df['impact'].map(impact_map)

    df['signal'] = df['z_score'] * df['impact_num']

    return df

pairs = ['eurusd', 'gbpusd', 'usdjpy']

dfs = [build_calendar_features(i) for i in pairs]

calendar_features = pd.concat(dfs).reset_index(drop= True)
calendar_features.drop_duplicates().reset_index(drop = True)

calendar_features.to_csv('data/processed_v2/calendar_features.csv', index= False)


In [9]:
eurusd = build_calendar_features('eurusd')
gbpusd = build_calendar_features('gbpusd')
usdjpy = build_calendar_features('usdjpy')

In [10]:
print(len(eurusd[eurusd['currency'] == 'EUR']))
print(len(eurusd[eurusd['currency'] == 'USD']))
print(len(gbpusd[gbpusd['currency'] == 'GBP']))
print(len(gbpusd[gbpusd['currency'] == 'USD']))
print(len(usdjpy[usdjpy['currency'] == 'JPY']))
print(len(usdjpy[usdjpy['currency'] == 'USD']))

4310
4942
2571
4942
1895
4942


In [11]:
eurusd[eurusd['currency'] == 'USD'].head()

,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal
5,2021-01-04,8:15pm,USD,Final Manufacturing PMI,Low,57.1,56.5,5.710000e+01,5.650000e+01,0.600,0.010619,1.245768,1,1.245768
6,2021-01-04,8:30pm,USD,Construction Spending m/m,Low,0.9%,1.6%,9.000000e-03,1.600000e-02,-0.007,-0.437500,-0.076648,1,-0.076648
12,2021-01-05,8:30pm,USD,ISM Manufacturing PMI,Medium,60.7,57.5,6.070000e+01,5.750000e+01,3.200,0.055652,1.978533,2,3.957066
13,2021-01-05,8:30pm,USD,ISM Manufacturing Prices,Low,77.6,65.4,7.760000e+01,6.540000e+01,12.200,0.186544,1.969510,1,1.969510
14,2021-01-05,All Day,USD,Wards Total Vehicle Sales,Low,16.3M,15.6M,1.630000e+07,1.560000e+07,700000.000,0.044872,0.652264,1,0.652264


In [12]:
gbpusd[gbpusd['currency'] == 'USD'].head()

,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal
4,2021-01-04,8:15pm,USD,Final Manufacturing PMI,Low,57.1,56.5,5.710000e+01,5.650000e+01,0.600,0.010619,0.789043,1,0.789043
5,2021-01-04,8:30pm,USD,Construction Spending m/m,Low,0.9%,1.6%,9.000000e-03,1.600000e-02,-0.007,-0.437500,-0.076648,1,-0.076648
6,2021-01-05,8:30pm,USD,ISM Manufacturing PMI,Medium,60.7,57.5,6.070000e+01,5.750000e+01,3.200,0.055652,1.978533,2,3.957066
7,2021-01-05,8:30pm,USD,ISM Manufacturing Prices,Low,77.6,65.4,7.760000e+01,6.540000e+01,12.200,0.186544,1.969510,1,1.969510
8,2021-01-05,All Day,USD,Wards Total Vehicle Sales,Low,16.3M,15.6M,1.630000e+07,1.560000e+07,700000.000,0.044872,0.652264,1,0.652264


In [13]:
usdjpy[usdjpy['currency'] == 'USD'].head()

,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal
1,2021-01-04,8:15pm,USD,Final Manufacturing PMI,Low,57.1,56.5,5.710000e+01,5.650000e+01,0.600,0.010619,0.954142,1,0.954142
2,2021-01-04,8:30pm,USD,Construction Spending m/m,Low,0.9%,1.6%,9.000000e-03,1.600000e-02,-0.007,-0.437500,-0.076648,1,-0.076648
4,2021-01-05,8:30pm,USD,ISM Manufacturing PMI,Medium,60.7,57.5,6.070000e+01,5.750000e+01,3.200,0.055652,1.978533,2,3.957066
5,2021-01-05,8:30pm,USD,ISM Manufacturing Prices,Low,77.6,65.4,7.760000e+01,6.540000e+01,12.200,0.186544,1.969510,1,1.969510
6,2021-01-05,All Day,USD,Wards Total Vehicle Sales,Low,16.3M,15.6M,1.630000e+07,1.560000e+07,700000.000,0.044872,0.652264,1,0.652264


In [4]:
cal_df = pd.read_csv("data/processed_v2/calendar_features.csv")
cal_df.head(3)

,date,time,currency,event,impact,actual,previous,actual_clean,previous_clean,change,change_rel,z_score,impact_num,signal
0,2021-01-04,1:45pm,EUR,Spanish Manufacturing PMI,Low,51.0,49.8,51.0,49.8,1.2,0.024096,0.688935,1,0.688935
1,2021-01-04,2:15pm,EUR,Italian Manufacturing PMI,Low,52.8,51.5,52.8,51.5,1.3,0.025243,0.729711,1,0.729711
2,2021-01-04,2:20pm,EUR,French Final Manufacturing PMI,Low,51.1,51.1,51.1,51.1,0.0,0.000000,-0.352682,1,-0.352682


In [3]:
cal_df.shape

(23602, 14)